# Lab 4: Deploy Medical AI Agent

AgentCore Runtime에 Medical AI Agent를 배포하고 Streamlit UI로 연결합니다.

## Architecture
```
User (Browser) → Streamlit (EC2:8501)
  → boto3 InvokeAgentRuntime
    → AgentCore Runtime (medical_agent.py + Strands + Bedrock Claude Sonnet 4.6)
      → AgentCore Gateway (OAuth) → MCP Server (Lambda) → EMR Serverless → S3 Tables
      → AgentCore Memory (STM + LTM)
      → AgentCore Observability (CloudWatch + X-Ray)
```

## Step 1: Configuration

In [ ]:
import boto3
import json
import time
import os
import zipfile
import io

REGION = (
    boto3.session.Session().region_name
    or os.environ.get("AWS_DEFAULT_REGION")
    or os.environ.get("AWS_REGION")
)
sts = boto3.client("sts")
ACCOUNT_ID = sts.get_caller_identity()["Account"]
AGENT_NAME = "fhir_medical_agent"
S3_BUCKET = f"fhir-data-{ACCOUNT_ID}-{REGION}"

# Load Gateway connection info from Lab 2
with open("connection_info.json") as f:
    conn = json.load(f)

GATEWAY_MCP_URL = conn["mcp_server_url"]
GATEWAY_CLIENT_ID = conn["client_id"]
GATEWAY_CLIENT_SECRET = conn["client_secret"]
GATEWAY_TOKEN_URL = conn["token_url"]

iam_client = boto3.client("iam")
s3_client = boto3.client("s3", region_name=REGION)
agentcore = boto3.client("bedrock-agentcore-control", region_name=REGION)

print(f"Account: {ACCOUNT_ID}")
print(f"Gateway: {GATEWAY_MCP_URL}")

## Step 2: Enable CloudWatch Transaction Search (Observability)

AgentCore Observability를 위해 CloudWatch Transaction Search를 활성화합니다. (계정당 1회)

In [ ]:
logs_client = boto3.client("logs", region_name=REGION)
xray_client = boto3.client("xray", region_name=REGION)

try:
    policy_doc = json.dumps({
        "Version": "2012-10-17",
        "Statement": [{
            "Sid": "TransactionSearchXRayAccess",
            "Effect": "Allow",
            "Principal": {"Service": "xray.amazonaws.com"},
            "Action": "logs:PutLogEvents",
            "Resource": [
                f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:aws/spans:*",
                f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:/aws/application-signals/data:*"
            ],
            "Condition": {
                "ArnLike": {"aws:SourceArn": f"arn:aws:xray:{REGION}:{ACCOUNT_ID}:*"},
                "StringEquals": {"aws:SourceAccount": ACCOUNT_ID}
            }
        }]
    })
    logs_client.put_resource_policy(policyName="AgentCoreTransactionSearch", policyDocument=policy_doc)
    print("✅ CloudWatch resource policy created")
except Exception as e:
    print(f"Resource policy: {e}")

try:
    xray_client.update_trace_segment_destination(Destination="CloudWatchLogs")
    print("✅ Transaction Search enabled")
except Exception as e:
    print(f"Transaction Search: {e}")

## Step 3: Create AgentCore Memory (STM + LTM)

에이전트의 대화 기억을 위한 Memory 리소스를 생성합니다.

In [ ]:
import uuid

memory_response = agentcore.create_memory(
    name=f"fhir_agent_mem_{uuid.uuid4().hex[:8]}",
    eventExpiryDuration=30,
    memoryStrategies=[
        {"userPreferenceMemoryStrategy": {
            "name": "user_prefs",
            "namespaces": ["/users/preferences/"]
        }},
        {"semanticMemoryStrategy": {
            "name": "medical_facts",
            "namespaces": ["/users/facts/"]
        }}
    ]
)

MEMORY_ID = memory_response["memory"]["id"]
print(f"Memory ID: {MEMORY_ID}")
print(f"Status: {memory_response['memory']['status']}")

# Wait for ACTIVE
print("Waiting for memory to become ACTIVE...")
for _ in range(60):
    mem_status = agentcore.get_memory(memoryId=MEMORY_ID)["memory"]["status"]
    if mem_status == "ACTIVE":
        print(f"✅ Memory is ACTIVE: {MEMORY_ID}")
        break
    time.sleep(5)
else:
    print(f"⚠️ Memory status: {mem_status}")

## Step 4: Create IAM Role for AgentCore Runtime

Runtime이 Bedrock 모델 호출, Memory 접근, Gateway 호출 등에 사용할 IAM Role을 생성합니다.

In [ ]:
RUNTIME_ROLE_NAME = "fhir-agentcore-runtime-role"

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
}

# Create role
try:
    role = iam_client.create_role(
        RoleName=RUNTIME_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="AgentCore Runtime role for FHIR Medical Agent"
    )
    print(f"Created role: {RUNTIME_ROLE_NAME}")
except iam_client.exceptions.EntityAlreadyExistsException:
    print(f"Role exists: {RUNTIME_ROLE_NAME}")

RUNTIME_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{RUNTIME_ROLE_NAME}"

# Attach permissions
runtime_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "BedrockModelAccess",
            "Effect": "Allow",
            "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"],
            "Resource": "*"
        },
        {
            "Sid": "MarketplaceForBedrock",
            "Effect": "Allow",
            "Action": ["aws-marketplace:ViewSubscriptions", "aws-marketplace:Subscribe", "aws-marketplace:Unsubscribe"],
            "Resource": "*"
        },
        {
            "Sid": "AgentCoreMemory",
            "Effect": "Allow",
            "Action": ["bedrock-agentcore:*"],
            "Resource": "*"
        },
        {
            "Sid": "S3CodeAccess",
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:GetBucketLocation"],
            "Resource": [
                f"arn:aws:s3:::{S3_BUCKET}",
                f"arn:aws:s3:::{S3_BUCKET}/*"
            ]
        },
        {
            "Sid": "CloudWatchLogs",
            "Effect": "Allow",
            "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents"],
            "Resource": "*"
        },
        {
            "Sid": "XRayTracing",
            "Effect": "Allow",
            "Action": ["xray:PutTraceSegments", "xray:PutTelemetryRecords"],
            "Resource": "*"
        },
        {
            "Sid": "ECRAccess",
            "Effect": "Allow",
            "Action": ["ecr:GetAuthorizationToken", "ecr:BatchGetImage", "ecr:GetDownloadUrlForLayer", "ecr:BatchCheckLayerAvailability"],
            "Resource": "*"
        }
    ]
}

iam_client.put_role_policy(
    RoleName=RUNTIME_ROLE_NAME,
    PolicyName="AgentCoreRuntimePolicy",
    PolicyDocument=json.dumps(runtime_policy)
)
print(f"✅ Role ARN: {RUNTIME_ROLE_ARN}")
time.sleep(10)  # Wait for IAM propagation


## Step 5: Build & Push Container Image

CodeBuild를 사용하여 에이전트 Docker 이미지를 빌드하고 ECR에 푸시합니다.

In [ ]:
# --- Step 5: Build container with CodeBuild and push to ECR ---
import zipfile, io

ECR_REPO_NAME = "fhir-medical-agent"
ecr_client = boto3.client("ecr", region_name=REGION)
codebuild_client = boto3.client("codebuild", region_name=REGION)

# 1. Create ECR repo
try:
    ecr_client.create_repository(repositoryName=ECR_REPO_NAME)
    print(f"✅ ECR repo created: {ECR_REPO_NAME}")
except ecr_client.exceptions.RepositoryAlreadyExistsException:
    print(f"ECR repo exists: {ECR_REPO_NAME}")

ECR_URI = f"{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/{ECR_REPO_NAME}:latest"
print(f"ECR URI: {ECR_URI}")

# 2. Upload agent source to S3 for CodeBuild
agent_dir = "../agent"
zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in ["medical_agent.py", "system_prompt.md", "system_prompt_legacy.md", "requirements.txt", "Dockerfile"]:
        filepath = os.path.join(agent_dir, fname)
        if os.path.exists(filepath):
            zf.write(filepath, fname)
            print(f"  Added: {fname}")
    # Add buildspec.yml inline
    buildspec = """version: 0.2
phases:
  pre_build:
    commands:
      - aws ecr get-login-password --region $AWS_DEFAULT_REGION | docker login --username AWS --password-stdin $ECR_REGISTRY
  build:
    commands:
      - docker build -t $ECR_REGISTRY/$ECR_REPO:latest .
  post_build:
    commands:
      - docker push $ECR_REGISTRY/$ECR_REPO:latest
"""
    zf.writestr("buildspec.yml", buildspec)

zip_buffer.seek(0)
s3_key = "agent-code/agent-source.zip"
s3_client.put_object(Bucket=S3_BUCKET, Key=s3_key, Body=zip_buffer.getvalue())
print(f"\n✅ Source uploaded to s3://{S3_BUCKET}/{s3_key}")

# 3. Create CodeBuild service role
CB_ROLE_NAME = "fhir-agent-codebuild-role"
cb_trust = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"Service": "codebuild.amazonaws.com"}, "Action": "sts:AssumeRole"}]
}
try:
    iam_client.create_role(RoleName=CB_ROLE_NAME, AssumeRolePolicyDocument=json.dumps(cb_trust))
except iam_client.exceptions.EntityAlreadyExistsException:
    pass

iam_client.put_role_policy(RoleName=CB_ROLE_NAME, PolicyName="CodeBuildPolicy", PolicyDocument=json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {"Effect": "Allow", "Action": ["ecr:*"], "Resource": "*"},
        {"Effect": "Allow", "Action": ["ecr:GetAuthorizationToken"], "Resource": "*"},
        {"Effect": "Allow", "Action": ["s3:GetObject", "s3:GetBucketLocation"], "Resource": [f"arn:aws:s3:::{S3_BUCKET}", f"arn:aws:s3:::{S3_BUCKET}/*"]},
        {"Effect": "Allow", "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents"], "Resource": "*"}
    ]
}))
CB_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{CB_ROLE_NAME}"
time.sleep(10)

# 4. Create/update CodeBuild project
CB_PROJECT = "fhir-medical-agent-build"
ECR_REGISTRY = f"{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com"
project_config = dict(
    name=CB_PROJECT,
    source={"type": "S3", "location": f"{S3_BUCKET}/{s3_key}"},
    artifacts={"type": "NO_ARTIFACTS"},
    environment={
        "type": "ARM_CONTAINER",
        "image": "aws/codebuild/amazonlinux2-aarch64-standard:3.0",
        "computeType": "BUILD_GENERAL1_SMALL",
        "privilegedMode": True,
        "environmentVariables": [
            {"name": "ECR_REGISTRY", "value": ECR_REGISTRY},
            {"name": "ECR_REPO", "value": ECR_REPO_NAME},
        ]
    },
    serviceRole=CB_ROLE_ARN,
)
try:
    codebuild_client.create_project(**project_config)
    print(f"✅ CodeBuild project created: {CB_PROJECT}")
except codebuild_client.exceptions.ResourceAlreadyExistsException:
    codebuild_client.update_project(**project_config)
    print(f"CodeBuild project updated: {CB_PROJECT}")

# 5. Start build
build = codebuild_client.start_build(projectName=CB_PROJECT)
build_id = build["build"]["id"]
print(f"\n🔨 Build started: {build_id}")
print("Waiting for build to complete (~2-3 min)...")

for i in range(60):
    b = codebuild_client.batch_get_builds(ids=[build_id])["builds"][0]
    bs = b["buildStatus"]
    if bs == "SUCCEEDED":
        print(f"\n✅ Build SUCCEEDED! Image: {ECR_URI}")
        break
    elif bs in ("FAILED", "FAULT", "STOPPED"):
        print(f"\n❌ Build {bs}")
        phases = b.get("phases", [])
        for p in phases:
            if p.get("phaseStatus") == "FAILED":
                print(f"  Failed phase: {p['phaseType']} - {p.get('contexts', [{}])[0].get('message', '')}")
        break
    print(f"  {bs} ({i*10}s)", end="\r")
    time.sleep(10)

## Step 6: Create AgentCore Runtime

에이전트를 AgentCore Runtime에 배포합니다.

In [ ]:
# Environment variables for the agent
env_vars = {
    "AWS_REGION": REGION,
    "BEDROCK_AGENTCORE_MEMORY_ID": MEMORY_ID,
    # "MODEL_ID": "us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    "MODEL_ID": "us.anthropic.claude-sonnet-4-6",
    "GATEWAY_MCP_URL": GATEWAY_MCP_URL,
    "GATEWAY_CLIENT_ID": GATEWAY_CLIENT_ID,
    "GATEWAY_CLIENT_SECRET": GATEWAY_CLIENT_SECRET,
    "GATEWAY_TOKEN_URL": GATEWAY_TOKEN_URL,
    "GATEWAY_SCOPE": "fhir-mcp/tools",
}

try:
    runtime = agentcore.create_agent_runtime(
        agentRuntimeName=AGENT_NAME,
        agentRuntimeArtifact={
            "containerConfiguration": {
                "containerUri": ECR_URI
            }
        },
        roleArn=RUNTIME_ROLE_ARN,
        networkConfiguration={"networkMode": "PUBLIC"},
        environmentVariables=env_vars,
        description="FHIR Medical AI Agent with MCP Gateway integration",
    )

    AGENT_RUNTIME_ARN = runtime["agentRuntimeArn"]
    AGENT_RUNTIME_ID = runtime["agentRuntimeId"]
    print(f"✅ Runtime created!")
    print(f"   ARN: {AGENT_RUNTIME_ARN}")
    print(f"   ID: {AGENT_RUNTIME_ID}")
    print(f"   Status: {runtime['status']}")

except agentcore.exceptions.ConflictException:
    # Already exists — update with new code
    runtimes = agentcore.list_agent_runtimes()
    for r in runtimes.get("agentRuntimeSummaries", []):
        if r["agentRuntimeName"] == AGENT_NAME:
            AGENT_RUNTIME_ARN = r["agentRuntimeArn"]
            AGENT_RUNTIME_ID = r["agentRuntimeId"]
            break
    print(f"Runtime exists, updating: {AGENT_RUNTIME_ID}")
    runtime = agentcore.update_agent_runtime(
        agentRuntimeId=AGENT_RUNTIME_ID,
        agentRuntimeArtifact={
            "containerConfiguration": {
                "containerUri": ECR_URI
            }
        },
        roleArn=RUNTIME_ROLE_ARN,
        networkConfiguration={"networkMode": "PUBLIC"},
        environmentVariables=env_vars,
    )
    print(f"✅ Runtime updated! Status: {runtime.get('status')}")

## Step 7: Wait for Runtime to be READY

In [ ]:
print("Waiting for runtime to become READY...")
for i in range(60):
    status = agentcore.get_agent_runtime(agentRuntimeId=AGENT_RUNTIME_ID)
    current = status["status"]
    if current == "READY":
        print(f"\n✅ Runtime is READY! (took ~{i*10}s)")
        break
    elif "FAILED" in current:
        print(f"\n❌ Runtime FAILED: {current}")
        print(json.dumps(status, indent=2, default=str))
        break
    print(f"  Status: {current} ({i*10}s elapsed)")
    time.sleep(10)
else:
    print(f"⚠️ Timeout. Current status: {current}")

## Step 8: Create Runtime Endpoint

In [ ]:
try:
    endpoint = agentcore.create_agent_runtime_endpoint(
        agentRuntimeId=AGENT_RUNTIME_ID,
        name="DEFAULT",
        agentRuntimeVersion=str(status.get("agentRuntimeVersion", "1")),
    )
    print(f"✅ Endpoint created")
except agentcore.exceptions.ConflictException:
    print("Endpoint already exists, skipping creation")

# Wait for endpoint
print("\nWaiting for endpoint to become READY...")
for i in range(60):
    ep = agentcore.get_agent_runtime_endpoint(
        agentRuntimeId=AGENT_RUNTIME_ID,
        endpointName="DEFAULT"
    )
    ep_status = ep["status"]
    if ep_status == "READY":
        print(f"\n✅ Endpoint READY! (took ~{i*10}s)")
        break
    elif "FAILED" in ep_status:
        print(f"\n❌ Endpoint FAILED: {ep_status}")
        break
    print(f"  Status: {ep_status} ({i*10}s elapsed)")
    time.sleep(10)

## Step 9: Test Agent — 기본 질문

In [ ]:
runtime_client = boto3.client("bedrock-agentcore", region_name=REGION)

def invoke_agent(prompt, session_id=None):
    import uuid
    if not session_id:
        session_id = str(uuid.uuid4()) + "-" + "a" * 10
    
    payload = json.dumps({"prompt": prompt}).encode("utf-8")
    
    response = runtime_client.invoke_agent_runtime(
        agentRuntimeArn=AGENT_RUNTIME_ARN,
        payload=payload,
        runtimeSessionId=session_id,
        contentType="application/json",
        accept="application/json",
    )
    
    raw = response["response"].read()
    content_type = response.get("contentType", "")
    
    if "application/json" in content_type:
        body = json.loads(raw)
        return body.get("response", str(body))
    return raw.decode("utf-8")

print("Testing: 테이블 목록 조회")
print("=" * 60)
result = invoke_agent("데이터 레이크에 어떤 테이블들이 있는지 알려줘")
print(result)

## Step 10: Test Agent — 환자 검색

In [ ]:
print("Testing: 당뇨 환자 검색")
print("=" * 60)
result = invoke_agent("오늘 외래에 당뇨 진단받은 60대 환자가 있는데, 목록 좀 보여줘.")
print(result)

## Step 11: Test Agent — 인구 건강 분석

In [ ]:
print("Testing: 인구 건강 분석")
print("=" * 60)
result = invoke_agent("60-69세 연령대의 당뇨병 환자 인구 건강 지표를 분석해줘")
print(result)

## Step 12: Test Memory — 세션 내 기억

In [ ]:
import uuid
session_id = str(uuid.uuid4()) + "-" + "a" * 10

r1 = invoke_agent("나는 내분비내과 전문의이고, 주로 당뇨병 환자를 담당하고 있어", session_id)
print("Message 1:", r1[:500])

r2 = invoke_agent("내가 담당하는 질환의 환자 통계를 보여줘", session_id)
print("\nMessage 2:", r2[:500])

## Step 13: Observability Dashboard

In [ ]:
print(f"📊 GenAI Observability Dashboard:")
print(f"   https://console.aws.amazon.com/cloudwatch/home?region={REGION}#gen-ai-observability/agent-core")

## Step 14: Run Frontend

AgentCore Runtime에 배포된 에이전트를 호출하는 React UI를 실행합니다.

### Security Group 설정

접속하는 브라우저의 IP 주소에 대해 8501, 3000번 포트를 오픈합니다.

브라우저에서 "What is my ip?"를 검색해서 현재 접속하는 IP 주소를 확보합니다.

IP 주소 확인 후 아래 셀을 실행하여, 해당 주소를 입력합니다.

In [ ]:
# 보안그룹에 8501, 3000 포트 추가 (접속자 IP만 허용)
ec2 = boto3.client('ec2', region_name=REGION)

my_ip = input('브라우저에서 https://checkip.amazonaws.com 접속 후 표시된 IP를 입력하세요: ').strip()

instances = ec2.describe_instances(
    Filters=[{'Name': 'tag:Name', 'Values': ['*CodeEditor*']}, {'Name': 'instance-state-name', 'Values': ['running']}]
)['Reservations']

if instances:
    sg_id = instances[0]['Instances'][0]['SecurityGroups'][0]['GroupId']
    public_ip = instances[0]['Instances'][0].get('PublicIpAddress', 'N/A')
    for port, label in [(8501, 'Streamlit UI'), (3000, 'React UI')]:
        try:
            ec2.authorize_security_group_ingress(
                GroupId=sg_id,
                IpPermissions=[{'IpProtocol': 'tcp', 'FromPort': port, 'ToPort': port, 'IpRanges': [{'CidrIp': f'{my_ip}/32', 'Description': label}]}]
            )
            print(f'✅ 보안그룹 {sg_id}에 {port} 포트 추가 완료 ({my_ip}/32)')
        except Exception as e:
            if 'Duplicate' in str(e):
                print(f'✅ {port} 포트 이미 허용됨')
            else:
                print(f'⚠️ {e}')
    print(f'\n🌐 Streamlit URL: http://{public_ip}:8501')
    print(f'🌐 React URL: http://{public_ip}:3000')
else:
    print('CodeEditor 인스턴스를 찾을 수 없습니다.')


### Frontend 구동 스크립트 실행

#### (Option-1) React App 구동 스크립트 실행
VS Code 터미널에서 **/workshop/building-medical-agent-from-ai-ready-datalake/run_react_app.sh** 스크립트를 실행합니다.

실행 후 위에 보이는 `React URL`을 클릭하여, 확인해봅니다.

#### (Option-2) Streamlit App 구동 스크립트 실행
VS Code 터미널에서 **/workshop/building-medical-agent-from-ai-ready-datalake/run_app.sh** 스크립트를 실행합니다.

실행 후 위에 보이는 `Streamlit URL`을 클릭하여, 확인해봅니다.

## Step 15: Save Agent Info

In [ ]:
agent_info = {
    "agent_runtime_arn": AGENT_RUNTIME_ARN,
    "agent_runtime_id": AGENT_RUNTIME_ID,
    "memory_id": MEMORY_ID,
    "gateway_mcp_url": GATEWAY_MCP_URL,
    "region": REGION,
}
with open("agent_info.json", "w") as f:
    json.dump(agent_info, f, indent=2)
print("✅ Saved to agent_info.json")
print(json.dumps(agent_info, indent=2))

## Cleanup (Optional)

In [ ]:
# ⚠️ Uncomment to delete resources
# agentcore.delete_agent_runtime_endpoint(agentRuntimeId=AGENT_RUNTIME_ID, endpointName="DEFAULT")
# agentcore.delete_agent_runtime(agentRuntimeId=AGENT_RUNTIME_ID)
# agentcore.delete_memory(memoryId=MEMORY_ID)
# iam_client.delete_role_policy(RoleName=RUNTIME_ROLE_NAME, PolicyName="AgentCoreRuntimePolicy")
# iam_client.delete_role(RoleName=RUNTIME_ROLE_NAME)
# print("All agent resources deleted")